In [1]:
import os
import pandas as pd
from uuid import uuid4
from tqdm.asyncio import tqdm
import re
from collections import defaultdict
import math
from typing import Optional, Tuple

In [2]:
from gecodb_compound_parser import Compound, perform_umlaut
from dereko_search import get_dereko_counts

In [ ]:
sp18_path = "../resources/schaefer_pankratz_2018/Concordance.full.csv"
celex_gml_path = "../resources/Celex/german/gml/gml.cd"
celex_gmw_path = "../resources/Celex/german/gmw/gmw.cd"

outdir = "../resources/developed/sp18"

outcsv_args = {
    "sep": "\t",
    "index": True,
    "header": True
}

In [4]:
sp18 = pd.read_csv(
    sp18_path,
    sep="\t",
    header=0,
    index_col="Match",
    dtype=defaultdict(lambda: str, LE="int"),   # all str, LE int
    quoting=3   # = csv.QUOTE_NONE, separate by tabs only
)
sp18.head()

,DocURL,DocID,SentID,LC,RC,N2Num,N2Typ,N1Lemma,N1Pl,LE
Match,,,,,,,,,,
Apfelästen,http://www.rattenforum.de/phpbb/viewtopic.php?...,3fc371483b113c1f9bbda743f88603d7014f,11,"Eben , als ich neues Futter und frisches Wasse...",waren einige und ganz vereinzelte an den Käfig...,Pl,Ind,Apfel,U,0
Apfelbad,http://old.glamour.de/glamour/7/2/content/0170...,552f4c720b020f5922a890c09ff8d60c2562,3,Dann spielen Sie mit uns und gewinnen Sie zwei...,relaxen ! Ihr Gutschein ist gültig bis Mai 200...,Sg,Ind,Apfel,U,0
Apfelbauer,http://www.genau-dein-obst.de/apfelblog/page/34,2a56dd938e65d789a0a9cca3015f17acdb16,44,Bei schönstem Sonnenschein nimmt Hannes Panter...,. Auf den Obstplantagen wachsen zum größten Te...,Sg,Ind,Apfel,U,0
Apfelbaum,http://blog.solingen-internet.de/totterblotsch...,fa540ade2bbfec78e9a3cb1a54706cd6da5e,60,Zuweilen gibt es aber auch romantische Momente...,""" für eine Hofschaft mit kleiner Straße nicht ...",Sg,Ind,Apfel,U,0
Apfelbäumchen,http://blog.wiwo.de/chefsache/2009/03/14/der-s...,8cd3866b0dd929e9bc7a40904fddb8ae0460,72,Manche Politiker reden ihre Anliegen ohne Not ...,und seinem diskontierten Wert nennt sich übrig...,Sg,Ind,Apfel,U,0


In [5]:
def umlambda(x: str) -> str:
    x = x.lower()
    x = re.sub("(?<![aeiouy])ae", "ä", x, flags=re.IGNORECASE)
    x = re.sub("(?<![aeiouy])oe", "ö", x, flags=re.IGNORECASE)
    x = re.sub("(?<![aeiouy])ue", "ü", x, flags=re.IGNORECASE)
    return x

In [6]:
gml = pd.read_csv(
    celex_gml_path,
    sep="\\",
    header=None,
    dtype=str,
    usecols=[0, 1],
    names=["id", "lemma"],
    index_col="id"
).dropna()

# keep only nouns
gml = gml[gml["lemma"].str[0].str.isupper()]

gml

,lemma
id,
1,A
2,Ae
4,Aal
6,Aar
7,Aas
...,...
51724,Zytologie
51725,Zytoplasma
51726,Zytosom


In [7]:
gmw = pd.read_csv(
    celex_gmw_path,
    sep="\\",
    header=None,
    dtype=str,
    usecols=[1, 3, 4],
    names=["wordform", "id", "forms"],
    index_col="id"
).dropna()

gmw

,wordform,forms
id,,
35,abbestelle,"1SIE,13SKE"
35,abbestellst,2SIE
35,abbestellen,"13PIE,13PKE,i"
35,abbestellte,"13SIA,13SKA"
35,abbestelltet,"2PIA,2PKA"
...,...,...
27964,Papste,dS
27964,Paepsten,dP
27964,Papstes,gS


In [8]:
# joining will automatically drop non-noun wordforms,
# because they have no lemma in gml
gmwl = gmw.join(gml["lemma"], how="inner")  \
    .reset_index(drop=True)  \
    .set_index("wordform")  \
    .sort_index()

gmwl.index = gmwl.index.map(umlambda)
gmwl["lemma"] = gmwl["lemma"].map(umlambda)
gmwl = gmwl[["lemma", "forms"]]

gmwl

,lemma,forms
wordform,,
a,a,"nP,gP,dP,aP,nS,gS,dS,aS"
aal,aal,"nS,dS,aS"
aale,aal,"nP,gP,aP,dS"
aalen,aal,dP
aales,aal,gS
...,...,...
schöbest kegel,kegelschieben,2SKA/
schöbet kegel,kegelschieben,2PKA/
schöbst kegel,kegelschieben,2SKA/


In [9]:
# Comparing the original Match column values with
# with reconstructions from `parse_sp18_comp()`
# (before lemmatization to NomSg with the help of CELEX2!)
# during development of this module
# revealed 154 mismatches in compounds built from
# six different N1. Manual inspection of these mismatches revealed that
# some compounds with N1s 'Dämon', 'Gerät', 'Geräusch', 'Schloss'
# were spelled alternatively, and that compounds
# with N1s 'Auge' and 'Birne' productively exhibit
# alternation between _+n_ and _-e_ (!).
# See more in 'resources/developed/sp18/leftovers/mismatches.csv'.
# Thus, we will correct
# the spellings of the first four N1s and remove
# all compounds with the last two N1s.

# After these two changes, all reconstructed compounds
# after `parse_sp18_comp()` do match the original Match column
# (before lemmatization to NomSg with the help of CELEX2!).

# correct spellings
for old, new in [
    ("Daemon", "Dämon"),
    ("Geraet", "Gerät"),
    ("Geraeusch", "Geräusch"),
    ("Schloß", "Schloss")
]:
    sp18.index = sp18.index.map(
        lambda x: x.replace(old, new)
        if x.startswith(old) else x
    )

# remove compounds with N1s 'Auge' (N=185) and 'Birne' (N=200)
sp18 = sp18[~sp18["N1Lemma"].isin(["Auge", "Birne"])]

In [10]:
# drop NaN (2 rows only)
sp18 = sp18.dropna()

In [11]:
# extract document metadata from the dataset;
# most probably, we won't need it (also
# many URLs are now down, anyways),
# so by decomposing the dataframe, we can
# reduce processing effort; on the other hand,
# joining the metadata back to the main dataframe
# is easy, so we can always do it later if needed

# first, join the left and right contexts into a single column,
# drop the original columns, and assign a unique id to each context
sp18["context"] = sp18.LC + " " + sp18.index + " " + sp18.RC
sp18.drop(columns=["LC", "RC"], inplace=True)
sp18["context_id"] = [str(uuid4()) for _ in range(len(sp18))]
sp18.head()

,DocURL,DocID,SentID,N2Num,N2Typ,N1Lemma,N1Pl,LE,context,context_id
Match,,,,,,,,,,
Apfelästen,http://www.rattenforum.de/phpbb/viewtopic.php?...,3fc371483b113c1f9bbda743f88603d7014f,11,Pl,Ind,Apfel,U,0,"Eben , als ich neues Futter und frisches Wasse...",8772a789-7093-44fc-a4d3-1a5b2a67b46f
Apfelbad,http://old.glamour.de/glamour/7/2/content/0170...,552f4c720b020f5922a890c09ff8d60c2562,3,Sg,Ind,Apfel,U,0,Dann spielen Sie mit uns und gewinnen Sie zwei...,15301504-bdb6-4f87-8466-5978755251c8
Apfelbauer,http://www.genau-dein-obst.de/apfelblog/page/34,2a56dd938e65d789a0a9cca3015f17acdb16,44,Sg,Ind,Apfel,U,0,Bei schönstem Sonnenschein nimmt Hannes Panter...,b3f54287-011d-497c-94ca-7dfea954ec6d
Apfelbaum,http://blog.solingen-internet.de/totterblotsch...,fa540ade2bbfec78e9a3cb1a54706cd6da5e,60,Sg,Ind,Apfel,U,0,Zuweilen gibt es aber auch romantische Momente...,7a917745-c1c7-4043-8a1b-d999caa50149
Apfelbäumchen,http://blog.wiwo.de/chefsache/2009/03/14/der-s...,8cd3866b0dd929e9bc7a40904fddb8ae0460,72,Sg,Ind,Apfel,U,0,Manche Politiker reden ihre Anliegen ohne Not ...,94b3ad67-62d5-40e6-9d66-5568738b999c


In [12]:
# now, extract the document metadata
# into a separate dataframe
sp18_meta = sp18[
        ["context_id", "DocID", "DocURL", "SentID"]
    ]   \
    .rename(
        columns={"DocID": "doc_id", "DocURL": "url", "SentID": "sent_n"}
    )   \
    .reset_index(drop=True) \
    .set_index("context_id")    \
    .sort_index()

sp18_meta.to_csv(
    os.path.join(outdir, "sp18_meta.tsv"),
    **outcsv_args
)
sp18_meta.head()

,doc_id,url,sent_n
context_id,,,
0014a118-7c97-4aca-a7e3-582d2fdf239c,b9aa1e59769538fecc5c985f4da3c0e6213f,http://www.thorwal-standard.de/rommilys/landru...,21
00169e54-b9ef-492c-a260-762aba173b36,e66d4f392dfa04c4fd540b37caeb6879cc60,http://www.hihawai.de/TravelMag/610/D%C3%A4nis...,9
00236f44-b8b5-4423-a330-3570b417cf1a,7e008b206b6268491af3b7cb1c72763de111,http://www.radioberlin.de/themen/mein_tagebuch...,304
003a30c2-c529-4711-b06f-8e1b88611c04,5e74dd31fdfd50d86cea6867d65d985dad10,http://www.forum-piercing.de/snakebites--und-q...,35
003bcfb2-d7c7-4fab-904e-4e1a61599ba7,bfa5764078165ea14d9d0800094572b20034,http://www.musikansich.de/ausgaben/1110/review...,7


In [13]:
# finally, drop the metadata columns
# from the main dataframe
sp18.drop(columns=["DocID", "DocURL", "SentID"], inplace=True)
sp18.head()

,N2Num,N2Typ,N1Lemma,N1Pl,LE,context,context_id
Match,,,,,,,
Apfelästen,Pl,Ind,Apfel,U,0,"Eben , als ich neues Futter und frisches Wasse...",8772a789-7093-44fc-a4d3-1a5b2a67b46f
Apfelbad,Sg,Ind,Apfel,U,0,Dann spielen Sie mit uns und gewinnen Sie zwei...,15301504-bdb6-4f87-8466-5978755251c8
Apfelbauer,Sg,Ind,Apfel,U,0,Bei schönstem Sonnenschein nimmt Hannes Panter...,b3f54287-011d-497c-94ca-7dfea954ec6d
Apfelbaum,Sg,Ind,Apfel,U,0,Zuweilen gibt es aber auch romantische Momente...,7a917745-c1c7-4043-8a1b-d999caa50149
Apfelbäumchen,Sg,Ind,Apfel,U,0,Manche Politiker reden ihre Anliegen ohne Not ...,94b3ad67-62d5-40e6-9d66-5568738b999c


In [14]:
# First, we need to be able to reconstruct the whole
# compound composition. We need that for three reasons:
# 1. To be able to revert the compounds to NomSg,
# which, in its turn, is required to identify duplicates;
# 2. Beside duplicates removal, we will also be able to
# identify contrastive pairs from the same constituents
# but with pluralic vs. non-pluralic linkers. This would
# be needed to assess the external plural effect
# following the method suggested in [pankratz_2017:20]
# (even though the examples are given for the internal plural effect).
# There is a change that the findings with this approach
# will be different as in [schaefer_pankratz_2018:26] since they
# do not contrast such pairs but rather compare all compounds in Pl forms
# to those in Sg forms.
# 3. Finally, similarly to [shmalts_2026], we would want
# to filter out the constituents not present in CELEX-nouns.

def lemmatize_n2(n2: str, expected_num: str) -> str:
    if n2 not in gmwl.index:
        return n2, False
    n2s = gmwl.loc[n2]
    if isinstance(n2s, pd.Series):  # one row found
        return n2s["lemma"], True
    else:
        if n2s.lemma.nunique() == 1:
            # one variant, ok
            return n2s.lemma.iloc[0], True
        else:
            # try finding a variant
            # that would correspond
            # to the annotated number of N2
            ncode = "P" if expected_num == "pl" else "S"
            matching_n2s = n2s[n2s["forms"].str.contains(ncode)]
            if matching_n2s.lemma.nunique() == 1:
                # one variant, ok
                return matching_n2s.lemma.iloc[0], True
    # otherwise: multiple variants matching the expected number,
    # unreliable parsing
    return n2, False


def parse_sp18_comp(
    comp: str,
    n1: str,
    pl_marker: str,
    has_linker: bool,
    n2_num: str,
    pbar: tqdm
) -> Tuple[str, str, str, str, bool]:

    pl_gc_template = "_+={pl_marker}_" if "U" in pl_marker else "_+{pl_marker}_"
    maybe_uml = perform_umlaut if ("U" in pl_marker and has_linker) else lambda x: x
    pl_marker = pl_marker.replace("U", "")
    # Schäfer & Pankratz 2018 only consider two
    # contrastive options: zero linker vs. pluralic linker;
    # are other non-pluralic linkers present in the dataset?
    linker = pl_marker if has_linker else ""
    n1_cstem = maybe_uml(n1) + linker
    n2 = re.sub(f"^{n1_cstem}", "", comp)

    # now, take the resulting n2 and lemmatize it to NomSg with the help of CELEX2
    n2, is_lemmatized = lemmatize_n2(n2, expected_num=n2_num)

    # gecodb style
    pl_gc = pl_gc_template.format(pl_marker=pl_marker)
    gecodb = f"{n1}{pl_gc if has_linker else '_'}{n2}"

    # recover lemma
    lemma = Compound(gecodb).lemma

    pbar.update(1)

    return gecodb, lemma, pl_gc, n2, is_lemmatized

In [15]:
sp18.index = sp18.index.map(str.lower)
for col in ["N1Lemma", "N2Num", "N2Typ"]:
    sp18[col] = sp18[col].map(str.lower)
sp18.head()

,N2Num,N2Typ,N1Lemma,N1Pl,LE,context,context_id
Match,,,,,,,
apfelästen,pl,ind,apfel,U,0,"Eben , als ich neues Futter und frisches Wasse...",8772a789-7093-44fc-a4d3-1a5b2a67b46f
apfelbad,sg,ind,apfel,U,0,Dann spielen Sie mit uns und gewinnen Sie zwei...,15301504-bdb6-4f87-8466-5978755251c8
apfelbauer,sg,ind,apfel,U,0,Bei schönstem Sonnenschein nimmt Hannes Panter...,b3f54287-011d-497c-94ca-7dfea954ec6d
apfelbaum,sg,ind,apfel,U,0,Zuweilen gibt es aber auch romantische Momente...,7a917745-c1c7-4043-8a1b-d999caa50149
apfelbäumchen,sg,ind,apfel,U,0,Manche Politiker reden ihre Anliegen ohne Not ...,94b3ad67-62d5-40e6-9d66-5568738b999c


In [16]:
with tqdm(total=len(sp18), desc="Lemmatizing compounds") as pbar:
    sp18["gecodb"], sp18["lemma"], sp18["n1_pl_marker"],    \
    sp18["n2"], sp18["is_lemmatized"] = zip(
        *sp18.apply(
            lambda row: parse_sp18_comp(
                comp=row.name,
                n1=row["N1Lemma"],
                pl_marker=row["N1Pl"],
                has_linker=int(row["LE"]),
                n2_num=row["N2Num"],
                pbar=pbar
            ),
            axis=1
        )
    )

sp18.head()

Lemmatizing compounds: 100%|██████████| 9028/9028 [00:13<00:00, 689.14it/s]


,N2Num,N2Typ,N1Lemma,N1Pl,LE,context,context_id,gecodb,lemma,n1_pl_marker,n2,is_lemmatized
Match,,,,,,,,,,,,
apfelästen,pl,ind,apfel,U,0,"Eben , als ich neues Futter und frisches Wasse...",8772a789-7093-44fc-a4d3-1a5b2a67b46f,apfel_ast,apfelast,_+=_,ast,True
apfelbad,sg,ind,apfel,U,0,Dann spielen Sie mit uns und gewinnen Sie zwei...,15301504-bdb6-4f87-8466-5978755251c8,apfel_bad,apfelbad,_+=_,bad,True
apfelbauer,sg,ind,apfel,U,0,Bei schönstem Sonnenschein nimmt Hannes Panter...,b3f54287-011d-497c-94ca-7dfea954ec6d,apfel_bauer,apfelbauer,_+=_,bauer,True
apfelbaum,sg,ind,apfel,U,0,Zuweilen gibt es aber auch romantische Momente...,7a917745-c1c7-4043-8a1b-d999caa50149,apfel_baum,apfelbaum,_+=_,baum,True
apfelbäumchen,sg,ind,apfel,U,0,Manche Politiker reden ihre Anliegen ohne Not ...,94b3ad67-62d5-40e6-9d66-5568738b999c,apfel_bäumchen,apfelbäumchen,_+=_,bäumchen,True


In [17]:
# Retrieve word counts from DeReKo; we have two scenarios:
# Lemmatized compounds may be searched directly;
# Non-lemmatized compounds will be first searched as a simple text search;
# if there are matches, the lemma annotation will be retrieved
# from the first match, and a consequent search for the retrieved lemma
# will be performed. This, the search will accomplish two goals:
# 1. It will allow us to retrieve counts for both lemmatized compounds
# (total word count of all wordforms) and counts for non-lemmatized compounds
# (only total count of the corresponding wordform).
# 2. It will partly retrieve the lemma annotation for non-lemmatized compounds.


def dereko_wrapper(
    input_lemmata: list[str],
    batch_size: Optional[int]=1000,
    **kwargs
) -> Tuple[list[int], list[str]]:

    # get dereko WCs; run in batches to enforce regular caching;
    # the cache is stored in 
    # resources/comp_constr_corp_study_mar26/dereko_de_geq70_counts.tsv
    freqs = []
    output_lemmata = []

    progress_bar = tqdm(
        total=math.ceil(len(input_lemmata) / batch_size),
        desc="Retrieving KorAP frequencies in batches"
    )

    for batch_start in range(0, len(input_lemmata), batch_size):
        batch = input_lemmata[batch_start:batch_start + batch_size]
        batch_freqs, batch_lemmata = get_dereko_counts(batch, **kwargs)
        freqs.extend(batch_freqs)
        output_lemmata.extend(batch_lemmata)
        progress_bar.update(1)
        
    return freqs, output_lemmata

In [ ]:
# get counts for lemmatized compounds
input_lemmata_l = sp18[sp18["is_lemmatized"]].lemma \
    .map(str.capitalize).unique().tolist()

freqs_l, _ = dereko_wrapper(
    input_lemmata_l,
    resolve_sz=True,
    is_base=True
)

In [ ]:
# get counts for non-lemmatized compounds
input_lemmata_nl = sp18[~sp18["is_lemmatized"]].lemma \
    .map(str.capitalize).unique().tolist()

freqs_l, output_lemmata = dereko_wrapper(
    input_lemmata_nl,
    resolve_sz=True,
    is_base=False
)

In [18]:
sp18 = sp18[
        [
            "gecodb", "lemma", "N1Lemma", "n2", "n1_pl_marker",
            "N2Typ", "N2Num", "LE", "is_lemmatized",
            "context", "context_id"
        ]
    ]   \
    .rename(
        columns={
            "N1Lemma": "n1", "N2Typ": "n1_sem",
            "N2Num": "n2_num", "LE": "is_linker_pl"
        }
    )   \
    .reset_index(drop=True) \
    .set_index("gecodb")    \
    .sort_index()

sp18.sample(5)

,lemma,n1,n2,n1_pl_marker,n1_sem,n2_num,is_linker_pl,is_lemmatized,context,context_id
gecodb,,,,,,,,,,
vogel_denkmal,vogeldenkmal,vogel,denkmal,_+=_,ind,pl,0,True,mit Joan Miró wird er im Frühling 1926 von Ser...,e081c4f9-cac7-4168-a35d-12aef84bb2fa
bett_+en_trakt,bettentrakt,bett,trakt,_+en_,konvc,sg,1,True,Katalog Peintures / Malerei Die Buchhandelsaus...,99cc102c-30e7-48ed-9fa5-6d1b72f5f8b5
ohr_+en_zeuge,ohrenzeuge,ohr,zeuge,_+en_,ind,sg,1,False,"Dies gilt verstärkt , je mehr Geltung der Vers...",5499a653-0c50-43af-922f-174d2ea3a741
bucht_+en_gemeinschaft,buchtengemeinschaft,bucht,gemeinschaft,_+en_,ind,sg,1,True,Bei konventioneller Spaltenbodenhaltung sind S...,76ac7060-df3d-4834-82b2-614252809d79
apfel_+=_korb,äpfelkorb,apfel,korb,_+=_,konvc,sg,1,True,"sage ich , die Mädchen hätten sich so schalkha...",81e8e9ce-b1f1-4d9e-bef3-53924d67a691


In [19]:
# From here on, we will have two versions of the dataset
# because reliable deduplication and detection of contrastive Sg-Pl pairs
# depends on the lemmatization of N2:
# 1. A version with lemmatized compounds only. We will extract the
# contrastive pairs from this version and use
# it for the analysis of the external plural effect.
# The compounds remaining after this extraction
# will be deduplicated and used for the analysis of
# the internal plural effect.
# 2. A joint version with both lemmatized and not lemmatized compounds.
# Will be used for the analysis of the internal plural effect only.
# Caution should be exercised when interpreting the results
# of the internal plural effect analysis based on the joint version.

# not-lemmatized compounds (N=1266)
sp18_nlemmatized = sp18[sp18["is_lemmatized"] == False]
sp18_nlemmatized.drop(columns=["is_lemmatized"], inplace=True)
sp18_nlemmatized.to_csv(
    os.path.join(outdir, "leftovers/sp18_nlemmatized.tsv"),
    **outcsv_args
)

# joint version (N=9028)
sp18_all = sp18.copy()
sp18_all.to_csv(
    os.path.join(outdir, "sp18_all.tsv"),
    **outcsv_args
)

# default version: lemmatized compounds only (N=7762)
sp18 = sp18_all[sp18_all["is_lemmatized"] == True]
sp18.drop(columns=["is_lemmatized"], inplace=True)
sp18.to_csv(
    os.path.join(outdir, "sp18.tsv"),
    **outcsv_args
)

In [20]:
# extract contrastive Sg-Pl pairs from the lemmatized version
sp18["const_comb"] = sp18["n1"] + "+" + sp18["n2"]
ident_const_all = sp18[sp18.duplicated(subset="const_comb", keep=False)]    \
    .sort_values("const_comb")  \
    .reset_index(drop=False)    \
    .set_index("const_comb")

ident_const_all.to_csv(
    os.path.join(outdir, "external_effect/ident_const_all.tsv"),
    **outcsv_args
)
ident_const_all

,gecodb,lemma,n1,n2,n1_pl_marker,n1_sem,n2_num,is_linker_pl,context,context_id
const_comb,,,,,,,,,,
apfel+bauer,apfel_+=_bauer,äpfelbauer,apfel,bauer,_+=_,ind,pl,1,"Kauf dir ein Apfel , und zahle Applepatente gl...",0b50211c-1d09-4751-9c12-94bba4f52975
apfel+bauer,apfel_bauer,apfelbauer,apfel,bauer,_+=_,ind,sg,0,Bei schönstem Sonnenschein nimmt Hannes Panter...,b3f54287-011d-497c-94ca-7dfea954ec6d
apfel+baum,apfel_+=_baum,äpfelbaum,apfel,baum,_+=_,ind,pl,1,Die Verlegung auf diesen Termin aus Witterungs...,163d941f-2637-4a3e-9435-5257d50d9eed
apfel+baum,apfel_+=_baum,äpfelbaum,apfel,baum,_+=_,ind,pl,1,Wir fahren an großen Obstgärten vorüber . Die ...,a303ac1b-df91-4652-812d-3656c3c7d330
apfel+baum,apfel_baum,apfelbaum,apfel,baum,_+=_,ind,pl,0,Land - und Forstwirtschaft überziehen große Re...,4a80df2a-7db1-4ce0-9f62-a7390284ac38
...,...,...,...,...,...,...,...,...,...,...
zitat+zitat,zitat_zitat,zitatzitat,zitat,zitat,_+e_,ind,pl,0,Hier fließt also ein Engagement in der Kunst m...,2575a481-1352-48a1-ac62-868f71109be8
zitat+zitat,zitat_+e_zitat,zitatezitat,zitat,zitat,_+e_,ind,sg,1,Immerhin werden hier scheinbar auch Menschen g...,ac48d9ab-bed0-4d02-8247-3c3e99356dca
zitat+zweck,zitat_zweck,zitatzweck,zitat,zweck,_+e_,ind,pl,0,"Ein rechtmäßiger Zitatzweck liegt nur vor , we...",453ed357-bc40-4539-a1a4-9975784c4b99


In [21]:
# First, we need to leave out compounds that do 
# not exhibit an alternation of Sg vs. Pl forms,
# that is, always occurred in either Sg or Pl form, but not in both.

ic_nums_data = {
    const_comb: {
        "sg_form": len(
            group[group["n2_num"] == "sg"]
        ),
        "pl_form": len(
            group[group["n2_num"] == "pl"]
        )
    }
    for const_comb, group in ident_const_all.groupby("const_comb")
}

ic_nums = pd.DataFrame.from_dict(ic_nums_data, orient="index")
# ic_nums.to_csv(
#     os.path.join(outdir, "external_effect/ic_nums.tsv"),
#     **outcsv_args
# )
ic_nums

,sg_form,pl_form
apfel+bauer,1,1
apfel+baum,2,5
apfel+berg,1,1
apfel+bäumchen,2,0
apfel+dieb,1,1
...,...,...
zitat+text,1,2
zitat+verwertung,1,1
zitat+verzeichnis,1,1
zitat+zitat,1,1


In [22]:
# only Sg forms (N=832)
ic_nalt_sg = ic_nums[ic_nums["pl_form"] == 0].index
ident_const_all.loc[ic_nalt_sg].to_csv(
    os.path.join(outdir, "external_effect/ic_nalt_sg.tsv"),
    **outcsv_args
)

# only Pl forms (N=219)
ic_nalt_pl = ic_nums[ic_nums["sg_form"] == 0].index
ident_const_all.loc[ic_nalt_pl].to_csv(
    os.path.join(outdir, "external_effect/ic_nalt_pl.tsv"),
    **outcsv_args
)

# alternating forms (N=2129)
ident_const_alt = ident_const_all[
    ~ident_const_all.index.isin(ic_nalt_sg) &
    ~ident_const_all.index.isin(ic_nalt_pl)
]
ident_const_alt.to_csv(
    os.path.join(outdir, "external_effect/ic_alt_all.tsv"),
    **outcsv_args
)
ident_const_alt

,gecodb,lemma,n1,n2,n1_pl_marker,n1_sem,n2_num,is_linker_pl,context,context_id
const_comb,,,,,,,,,,
apfel+bauer,apfel_+=_bauer,äpfelbauer,apfel,bauer,_+=_,ind,pl,1,"Kauf dir ein Apfel , und zahle Applepatente gl...",0b50211c-1d09-4751-9c12-94bba4f52975
apfel+bauer,apfel_bauer,apfelbauer,apfel,bauer,_+=_,ind,sg,0,Bei schönstem Sonnenschein nimmt Hannes Panter...,b3f54287-011d-497c-94ca-7dfea954ec6d
apfel+baum,apfel_+=_baum,äpfelbaum,apfel,baum,_+=_,ind,pl,1,Die Verlegung auf diesen Termin aus Witterungs...,163d941f-2637-4a3e-9435-5257d50d9eed
apfel+baum,apfel_+=_baum,äpfelbaum,apfel,baum,_+=_,ind,pl,1,Wir fahren an großen Obstgärten vorüber . Die ...,a303ac1b-df91-4652-812d-3656c3c7d330
apfel+baum,apfel_baum,apfelbaum,apfel,baum,_+=_,ind,pl,0,Land - und Forstwirtschaft überziehen große Re...,4a80df2a-7db1-4ce0-9f62-a7390284ac38
...,...,...,...,...,...,...,...,...,...,...
zitat+zitat,zitat_zitat,zitatzitat,zitat,zitat,_+e_,ind,pl,0,Hier fließt also ein Engagement in der Kunst m...,2575a481-1352-48a1-ac62-868f71109be8
zitat+zitat,zitat_+e_zitat,zitatezitat,zitat,zitat,_+e_,ind,sg,1,Immerhin werden hier scheinbar auch Menschen g...,ac48d9ab-bed0-4d02-8247-3c3e99356dca
zitat+zweck,zitat_zweck,zitatzweck,zitat,zweck,_+e_,ind,pl,0,"Ein rechtmäßiger Zitatzweck liegt nur vor , we...",453ed357-bc40-4539-a1a4-9975784c4b99


In [23]:
# We assume that since in standard (not colloquial)
# German, the combination of N1 and the linker should
# remain the same in Sg and Pl compound form,
# (e.g. Apfelbaum vs. Apfelbäume),
# and since the internal plural effect should enforce
# the pluralic linker, nouns that undergo the
# internal plural effect may not exhibit a reliable
# alternation in Sg vs. Pl forms, since they would
# receive the pluralic linker in its base form (Sg)
# and bring it to the Pl form. This is exemplified 
# by the examples in the dataset of Äpfelkiste vs. Äpfelkisten,
# or Liederabend vs. Liederabende.
# We therefore further separate the data onto the subsets
# of alternating compounds with no internal plural effect
# and those with the internal plural effect.

# internal plural effect (N=341)
ic_alt_ipe = ident_const_alt[
    ~(ident_const_alt["n1_sem"] == "ind")
]
ic_alt_ipe.to_csv(
    os.path.join(outdir, "external_effect/ic_alt_ipe.tsv"),
    **outcsv_args
)

# internal plural effect (N=1788)
ic_alt_nipe = ident_const_alt[
    ident_const_alt["n1_sem"] == "ind"
]
ic_alt_nipe.to_csv(
    os.path.join(outdir, "external_effect/ic_alt_nipe.tsv"),
    **outcsv_args
)
ic_alt_nipe

,gecodb,lemma,n1,n2,n1_pl_marker,n1_sem,n2_num,is_linker_pl,context,context_id
const_comb,,,,,,,,,,
apfel+bauer,apfel_+=_bauer,äpfelbauer,apfel,bauer,_+=_,ind,pl,1,"Kauf dir ein Apfel , und zahle Applepatente gl...",0b50211c-1d09-4751-9c12-94bba4f52975
apfel+bauer,apfel_bauer,apfelbauer,apfel,bauer,_+=_,ind,sg,0,Bei schönstem Sonnenschein nimmt Hannes Panter...,b3f54287-011d-497c-94ca-7dfea954ec6d
apfel+baum,apfel_+=_baum,äpfelbaum,apfel,baum,_+=_,ind,pl,1,Die Verlegung auf diesen Termin aus Witterungs...,163d941f-2637-4a3e-9435-5257d50d9eed
apfel+baum,apfel_+=_baum,äpfelbaum,apfel,baum,_+=_,ind,pl,1,Wir fahren an großen Obstgärten vorüber . Die ...,a303ac1b-df91-4652-812d-3656c3c7d330
apfel+baum,apfel_baum,apfelbaum,apfel,baum,_+=_,ind,pl,0,Land - und Forstwirtschaft überziehen große Re...,4a80df2a-7db1-4ce0-9f62-a7390284ac38
...,...,...,...,...,...,...,...,...,...,...
zitat+zitat,zitat_zitat,zitatzitat,zitat,zitat,_+e_,ind,pl,0,Hier fließt also ein Engagement in der Kunst m...,2575a481-1352-48a1-ac62-868f71109be8
zitat+zitat,zitat_+e_zitat,zitatezitat,zitat,zitat,_+e_,ind,sg,1,Immerhin werden hier scheinbar auch Menschen g...,ac48d9ab-bed0-4d02-8247-3c3e99356dca
zitat+zweck,zitat_zweck,zitatzweck,zitat,zweck,_+e_,ind,pl,0,"Ein rechtmäßiger Zitatzweck liegt nur vor , we...",453ed357-bc40-4539-a1a4-9975784c4b99


In [24]:
# Now furthermore, as we know, semantic constraints
# are not the only constraints that govern linker choice.
# There may be cases where a constituent
# with no internal plural effect may still consistently 
# receive a pluralic linker (with no plural cue) due to other constraints,
# e.g. phonological or morphological. Since a compound with a pluralic linker
# in Sg will most probably appear with the same linker in Pl
# (see the explanation above), we expect to see external effect (if it exists)
# only in the compounds that usually take a zero linker in Sg form.
# Otherwise, if the compound usually takes a pluralic linker in Sg form
# (be it due to the internal plural effect or other constraints),
# the chances are that it will also take a pluralic linker in Pl form, and thus
# the external plural effect will be masked. This is exemplified
# by the examples in the dataset of Katzenfell vs. Katzenfelle,
# Kinderbuch vs. Kinderbücher.
# Therefore, we will here separate the subset into the group
# of compounds that mostly take a zero linker in Sg form
# and the group of compounds that mostly take a pluralic linker in Sg form.

ic_alt_nipe_sg_links_data = {
    const_comb: {
        "sg_l": len(
            group[
                (group["n2_num"] == "sg") &
                (~group["is_linker_pl"])
            ]
        ),
        "pl_l": len(
            group[
                (group["n2_num"] == "sg") &
                (group["is_linker_pl"])
            ]
        )
    }
    for const_comb, group in ic_alt_nipe.groupby("const_comb")
}

ic_alt_nipe_sg_links = pd.DataFrame.from_dict(ic_alt_nipe_sg_links_data, orient="index")
# ic_nums.to_csv(
#     os.path.join(outdir, "external_effect/ic_nums.tsv"),
#     **outcsv_args
# )
ic_alt_nipe_sg_links

,sg_l,pl_l
apfel+bauer,1,0
apfel+baum,2,0
apfel+dieb,0,1
apfel+hof,1,0
apfel+kopf,1,0
...,...,...
zitat+tag,1,0
zitat+text,0,1
zitat+verwertung,0,1
zitat+zitat,0,1


In [25]:
# mostly pluralic linker in the Sg form (N=569)
ic_alt_nipe_pl_l = ic_alt_nipe_sg_links[
    ic_alt_nipe_sg_links["pl_l"] >
    ic_alt_nipe_sg_links["sg_l"]
].index
ic_alt_nipe.loc[ic_alt_nipe_pl_l].to_csv(
    os.path.join(outdir, "external_effect/ic_alt_nipe_pl_l.tsv"),
    **outcsv_args
)

# mostly zero linker in the Sg form (N=1219)
ic_alt_nipe_sg_l = ic_alt_nipe[
    ~ic_alt_nipe.index.isin(ic_alt_nipe_pl_l)
]
ic_alt_nipe_sg_l.to_csv(
    os.path.join(outdir, "external_effect/ic_alt_nipe_sg_l.tsv"),
    **outcsv_args
)
ic_alt_nipe_sg_l

,gecodb,lemma,n1,n2,n1_pl_marker,n1_sem,n2_num,is_linker_pl,context,context_id
const_comb,,,,,,,,,,
apfel+bauer,apfel_+=_bauer,äpfelbauer,apfel,bauer,_+=_,ind,pl,1,"Kauf dir ein Apfel , und zahle Applepatente gl...",0b50211c-1d09-4751-9c12-94bba4f52975
apfel+bauer,apfel_bauer,apfelbauer,apfel,bauer,_+=_,ind,sg,0,Bei schönstem Sonnenschein nimmt Hannes Panter...,b3f54287-011d-497c-94ca-7dfea954ec6d
apfel+baum,apfel_+=_baum,äpfelbaum,apfel,baum,_+=_,ind,pl,1,Die Verlegung auf diesen Termin aus Witterungs...,163d941f-2637-4a3e-9435-5257d50d9eed
apfel+baum,apfel_+=_baum,äpfelbaum,apfel,baum,_+=_,ind,pl,1,Wir fahren an großen Obstgärten vorüber . Die ...,a303ac1b-df91-4652-812d-3656c3c7d330
apfel+baum,apfel_baum,apfelbaum,apfel,baum,_+=_,ind,pl,0,Land - und Forstwirtschaft überziehen große Re...,4a80df2a-7db1-4ce0-9f62-a7390284ac38
...,...,...,...,...,...,...,...,...,...,...
zitat+tag,zitat_tag,zitattag,zitat,tag,_+e_,ind,pl,0,"Du kannst dir die Textpassage , welche du ziti...",dbaa999c-ba4d-4fde-a5b8-80568261d403
zitat+tag,zitat_tag,zitattag,zitat,tag,_+e_,ind,sg,0,Der Vollständigkeit halber sollten Sie die Ang...,136b3472-a425-4d1c-930b-268fe2e9bcb1
zitat+zweck,zitat_zweck,zitatzweck,zitat,zweck,_+e_,ind,pl,0,"Ein rechtmäßiger Zitatzweck liegt nur vor , we...",453ed357-bc40-4539-a1a4-9975784c4b99


In [26]:
ic_alt_nipe_sg_l_sum_data = {}
for const_comb, group in ic_alt_nipe_sg_l.groupby("const_comb"):
    for num in ["sg", "pl"]:
        ic_alt_nipe_sg_l_sum_data[(const_comb, f"num_{num}")] = {
            "l_sg": len(
                group[(group["n2_num"] == num) & (~group["is_linker_pl"])]
            ),
            "l_pl": len(
                group[(group["n2_num"] == num) & (group["is_linker_pl"])]
            )
        }

ic_alt_nipe_sg_l_sum = pd.DataFrame(ic_alt_nipe_sg_l_sum_data).T
ic_alt_nipe_sg_l_sum["pl_share"] = ic_alt_nipe_sg_l_sum["l_pl"] / (
    ic_alt_nipe_sg_l_sum["l_sg"] + ic_alt_nipe_sg_l_sum["l_pl"]
)
# ic_alt_nipe_sg_l_sum.to_csv(
#     os.path.join(outdir, "external_effect/ic_alt_nipe_sg_l_sum.tsv"),
#     **outcsv_args
# )
ic_alt_nipe_sg_l_sum

l_sg  l_pl  pl_share
apfel+bauer   num_sg     1     0       0.0
              num_pl     0     1       1.0
apfel+baum    num_sg     2     0       0.0
              num_pl     2     3       0.6
apfel+hof     num_sg     1     0       0.0
...                    ...   ...       ...
zitat+spender num_pl     0     1       1.0
zitat+tag     num_sg     1     0       0.0
              num_pl     1     0       0.0
zitat+zweck   num_sg     2     0       0.0
              num_pl     1     0       0.0

[932 rows x 3 columns]

In [27]:
ic_alt_nipe_sg_l_shares = ic_alt_nipe_sg_l_sum.unstack()["pl_share"]
ic_alt_nipe_sg_l_shares = ic_alt_nipe_sg_l_shares[["num_sg", "num_pl"]]
ic_alt_nipe_sg_l_shares

,num_sg,num_pl
apfel+bauer,0.0,1.0
apfel+baum,0.0,0.6
apfel+hof,0.0,0.0
apfel+kopf,0.0,1.0
apfel+plantage,0.0,0.5
...,...,...
zitat+gedicht,0.0,0.0
zitat+recht,0.5,0.0
zitat+spender,0.0,1.0
zitat+tag,0.0,0.0


In [28]:
# A few constituent combinations (supposedly) exhibit
# variation between Sg and Pl-like meaning of N1
# in different entries depending on the context (refer to the dataset).
# These are:
# 'bad+ausstellung', 'bett+belegung', 'bucht+system', 'ei+paket',
# 'ei+sack', 'element+häufigkeit', 'element+zuordnung', 'gerät+verlust',
# 'gerät+wart', 'hand+kontakt', 'hund+buch', 'kommune+bewegung',
# 'kommune+tag', 'loch+dichte', 'nation+tafel', 'produkt+familie',
# 'schloss+tour', 'schwester+genossenschaft', 'stadt+tour', 'vogel+buch',
# 'vogel+zimmer', 'zahn+differenz'.
# Since this difference points to the internal rather than external plural effect,
# we exclude these combinations from the analysis of the external plural effect.
inconsistent_n1_sem = ic_alt_nipe_sg_l_shares[ic_alt_nipe_sg_l_shares.isna().any(axis=1)]
ic_alt_nipe_sg_l.loc[inconsistent_n1_sem.index].to_csv(
    os.path.join(outdir, "leftovers/inconsistent_n1_sem.tsv"),
    **outcsv_args
)

ic_alt_nipe_sg_l_shares.dropna(inplace=True)
ic_alt_nipe_sg_l = ic_alt_nipe_sg_l[
    ~ic_alt_nipe_sg_l.index.isin(inconsistent_n1_sem.index)
]

ic_alt_nipe_sg_l

,gecodb,lemma,n1,n2,n1_pl_marker,n1_sem,n2_num,is_linker_pl,context,context_id
const_comb,,,,,,,,,,
apfel+bauer,apfel_+=_bauer,äpfelbauer,apfel,bauer,_+=_,ind,pl,1,"Kauf dir ein Apfel , und zahle Applepatente gl...",0b50211c-1d09-4751-9c12-94bba4f52975
apfel+bauer,apfel_bauer,apfelbauer,apfel,bauer,_+=_,ind,sg,0,Bei schönstem Sonnenschein nimmt Hannes Panter...,b3f54287-011d-497c-94ca-7dfea954ec6d
apfel+baum,apfel_+=_baum,äpfelbaum,apfel,baum,_+=_,ind,pl,1,Die Verlegung auf diesen Termin aus Witterungs...,163d941f-2637-4a3e-9435-5257d50d9eed
apfel+baum,apfel_+=_baum,äpfelbaum,apfel,baum,_+=_,ind,pl,1,Wir fahren an großen Obstgärten vorüber . Die ...,a303ac1b-df91-4652-812d-3656c3c7d330
apfel+baum,apfel_baum,apfelbaum,apfel,baum,_+=_,ind,pl,0,Land - und Forstwirtschaft überziehen große Re...,4a80df2a-7db1-4ce0-9f62-a7390284ac38
...,...,...,...,...,...,...,...,...,...,...
zitat+tag,zitat_tag,zitattag,zitat,tag,_+e_,ind,pl,0,"Du kannst dir die Textpassage , welche du ziti...",dbaa999c-ba4d-4fde-a5b8-80568261d403
zitat+tag,zitat_tag,zitattag,zitat,tag,_+e_,ind,sg,0,Der Vollständigkeit halber sollten Sie die Ang...,136b3472-a425-4d1c-930b-268fe2e9bcb1
zitat+zweck,zitat_zweck,zitatzweck,zitat,zweck,_+e_,ind,pl,0,"Ein rechtmäßiger Zitatzweck liegt nur vor , we...",453ed357-bc40-4539-a1a4-9975784c4b99


In [91]:
from scipy.stats import chi2_contingency
from statsmodels.stats.contingency_tables import Table

In [69]:
res_outdir = "../out/external_effect"

outcsv_args = {
    "sep": "\t",
    "index": True,
    "header": True
}

In [70]:
ic_alt_nipe_sg_l_shares_desc = ic_alt_nipe_sg_l_shares.describe()
ic_alt_nipe_sg_l_shares_desc.to_csv(
    os.path.join(res_outdir, "ic_alt_nipe_sg_l/desc.tsv"),
    **outcsv_args
)
ic_alt_nipe_sg_l_shares_desc

,num_sg,num_pl
count,444.000000,444.000000
mean,0.112800,0.300526
std,0.205177,0.420830
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.000000,0.616667
max,0.500000,1.000000


In [71]:
# First, try the same methodology as in the
# original [schaefer_pankratz_2018:26] paper:
# 1. Create contingency table:
# n2_num (Sg vs. Pl) x n1_pl_marker (zero vs. pluralic linker).
# 2. Perform chi-squared test of independence over the table
# to assess the statistical significance of the association between
# the Pl form of N2 and the presence of a pluralic linker in N1.
# 3. Perform the Cramer's V test to assess the strength
# of the association.
# 4. Inspect symmetrical standardized residuals to identify
# the cells that contribute to the association, and in which direction.

# create contingency table replicating [schaefer_pankratz_2018:26]
ic_alt_nipe_sg_l_ct = ic_alt_nipe_sg_l_sum \
    .groupby(level=1)[["l_sg", "l_pl"]].sum()   \
    .T  \
    # [["num_sg", "num_pl"]]
ic_alt_nipe_sg_l_ct.to_csv(
    os.path.join(res_outdir, "ic_alt_nipe_sg_l/contingency_table.tsv"),
    **outcsv_args
)
ic_alt_nipe_sg_l_ct

,num_pl,num_sg
l_sg,401,513
l_pl,195,110


In [82]:
ic_alt_nipe_sg_l_ct.values.sum()

np.int64(1219)

In [ ]:
# even though I didn't find the original R code
# of [schaefer_pankratz_2018] for this experiment,
# the `chi2_contingency` function from `scipy.stats`
# produced the same results as in the paper
# on the same contingency table, so we can be
# confident that we are replicating the same analysis
chi2_res = chi2_contingency(ic_alt_nipe_sg_l_ct)

# the original R code for Cramer's V is available in
# linkingelements/Data/R/functions.R, and is as follows:
# 
#   cramer.v <- function(chisq) {
#       unname(sqrt(chisq$statistic/(sum(chisq$observed)*min(dim(chisq$observed)-1))))
#   }
#
# we will reconstruct it in Python:
#   dim(chisq$observed) -> ic_alt_nipe_sg_l_ct.shape
#
#   min(dim(chisq$observed)-1) -> min(
#       ic_alt_nipe_sg_l_ct.shape[0] - 1,
#       ic_alt_nipe_sg_l_ct.shape[1] - 1
#   )
# 
#   sum(chisq$observed) -> ic_alt_nipe_sg_l_ct.values.sum()
#
#   unname -> [irrelevant for Python]
def cramer_v(chi2: float, contingency_table: pd.DataFrame) -> float:
    return math.sqrt(
        chi2 / (
            contingency_table.values.sum() *
            min(
                contingency_table.shape[0] - 1,
                contingency_table.shape[1] - 1
            )
        )
    )

cramer_v_val = cramer_v(chi2_res.statistic, ic_alt_nipe_sg_l_ct)

# the original R code for Cohen's W is available in
# linkingelements/Data/R/functions.R, and is as follows:
# 
#   # cohen.w <- function(chisq) {
#       cramer.v(chisq) * sqrt(min(dim(chisq$observed)-1))
#   }
def cohen_w(cramer_v: float, contingency_table: pd.DataFrame) -> float:
    return cramer_v * math.sqrt(
        min(
            contingency_table.shape[0] - 1,
            contingency_table.shape[1] - 1
        )
    )

cohen_w_val = cohen_w(cramer_v_val, ic_alt_nipe_sg_l_ct)

ic_alt_nipe_sg_l_chi2 = pd.Series(
    data={
        "chi2": chi2_res.statistic.round(3),
        "df": chi2_res.dof,
        # = 1.9e-09, significant
        "p": chi2_res.pvalue,#.round(3),
        # = 0.172, small to medium effect size,
        # see ???
        "cramer_v": round(cramer_v_val, 3),
        # = 0.172, small to medium effect size,
        # see [cohen_1998:224-225]
        "cohen_w": round(cohen_w_val, 3)
    }
)

ic_alt_nipe_sg_l_chi2.to_csv(
    os.path.join(res_outdir, "ic_alt_nipe_sg_l/chi2.tsv"),
    **{**outcsv_args, "header": False}
)

ic_alt_nipe_sg_l_chi2

chi2        3.603400e+01
df          1.000000e+00
p           1.938667e-09
cramer_v    1.720000e-01
cohen_w     1.720000e-01
dtype: float64

In [ ]:
# even though I didn't find the original R code
# of [schaefer_pankratz_2018] for this experiment,
# the `standardized_resids` property of
# `statsmodels.stats.contingency_tables.Table`
# produced the same results as in the paper
# on the same contingency table, so we can be
# confident that we are replicating the same analysis
ic_alt_nipe_sg_l_resids = Table(ic_alt_nipe_sg_l_ct)    \
    .standardized_resids.round(3)
ic_alt_nipe_sg_l_resids.to_csv(
    os.path.join(res_outdir, "ic_alt_nipe_sg_l/resids.tsv"),
    **outcsv_args
)
# -6.069 for opposite number-linker: Pl vs non-pluralic and Sg vs pluralic
# 6.069 for congruent number-linker: Pl vs pluralic and Sg vs non-pluralic
# => 
ic_alt_nipe_sg_l_resids

,num_pl,num_sg
l_sg,-6.069,6.069
l_pl,6.069,-6.069
